# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Runner

**Phase 1.5: 10-Layer Cooperative Steering**

Pipeline: `Norm Calibration → S1 Training → S5 Calibration → Benchmark`

**Target:** 10-layer SteerNet achieves ≥ +15 pp zero-shot gain, non-negative 2-shot/4-shot delta.

**Phase 1.5 changes (on top of V3):**
- 10 candidate layers: `[8, 11, 13, 15, 17, 19, 20, 21, 22, 24]` — early/mid/late coverage
- Cooperative implicit steering via residual stream (each layer builds on previous)
- `mag_head.bias = -3.0` init — starts selective (5%), rises only where typed loss demands
- `mag_head.bias` in decay group — L2 prevents unbounded bias growth
- Typed L1 penalty on T_PROSE only — selective suppression, not uniform damping
- `mag_l1_weight = 0.20` (was 0.02) — effective against Adam's v_t accumulation
- `focal_gamma = 2.0` (was 1.0) — concentrates gradient on hard failures
- KL extended to T_OPERAND + T_OPERATOR (`kl_arithmetic_weight=0.30`) — 4-shot preservation
- `train_v5_demo_frac = 0.80` (was 0.50) + rotating exemplar selection — no memorization
- `steer_amplitude_k`: optional override for per-layer amplitude (default: None = use k=10)
- Conv cache order fixed: conv runs before cache update (safety fix)
- Strict regex handles negative final answers


## 1. Environment & Hardware Check

In [ ]:
# Set CUDA allocator config BEFORE any torch/transformers import.
# expandable_segments:True lets the allocator satisfy fragmented-memory requests
# without requiring a single contiguous free block — critical for 10-hook training.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy tqdm


## 2. Clone Repository (`development` branch)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")

# Phase 1.5: 10 candidate layers — always recalibrate to get accurate N_l for new layers.
# Calibration takes ~3-5 min on T4 and is required for correct per-layer norm caps.
NEW_LAYERS = '8 11 13 15 17 19 20 21 22 24'
print(f"Calibrating norms for 10-layer route: [{NEW_LAYERS}]")
!python scripts/calibrate_norms.py \
    --n_samples 500 \
    --output norms_calibration.json \
    --candidate_layers {NEW_LAYERS}

print("\n✅ Norm calibration complete:")
!ls -lh norms_calibration.json


## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache


In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration (Phase 1 V2 — Zero Heuristic Dials)


In [ ]:
# Phase 1.5 config — batch=2 re-enabled after PYTORCH_CUDA_ALLOC_CONF fix
PROFILE = 'balanced'
S1_OBJECTIVE = 'nll'
LOSS_MODE = 'typed'
STEER_KAPPA  = 0.04
K_SCALE_MODE = 'sqrt'
STEER_SCALE  = 0.30

# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True is now set correctly (cell 1).
# With expandable segments the allocator handles fragmented 100MB->34MB requests.
# batch=2 re-enabled: estimated peak ~11-12 GB on T4 14.56 GB.
# Fallback if still OOM: BATCH_SIZE=1, GRAD_ACCUM=16
BATCH_SIZE = 2
GRAD_ACCUM = 8    # effective batch = 16
MAX_TOK_LEN = 1024

if PROFILE == 'quick':
    S1_EPOCHS=2;  S1_LR=2e-4;  S1_TRAIN_N=50;  S1_VAL_N=10; CALIB_SEL_N=5;  BENCH_N_TEST=10;  BENCH_SHOTS='0'
elif PROFILE == 'balanced':
    S1_EPOCHS=5;  S1_LR=2e-4;  S1_TRAIN_N=3000; S1_VAL_N=100; CALIB_SEL_N=60; BENCH_N_TEST=300; BENCH_SHOTS='0 2 4'
elif PROFILE == 'medium':
    S1_EPOCHS=5;  S1_LR=2e-4;  S1_TRAIN_N=3000; S1_VAL_N=100; CALIB_SEL_N=60; BENCH_N_TEST=100; BENCH_SHOTS='0 2 4'
elif PROFILE == 'full':
    S1_EPOCHS=5;  S1_LR=2e-4;  S1_TRAIN_N=7473; S1_VAL_N=120; CALIB_SEL_N=120; BENCH_N_TEST=500; BENCH_SHOTS='0 2 4'

print(f'Profile={PROFILE} | batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} | epochs={S1_EPOCHS}')
print(f'Target: 0-shot >= +15pp | 2-shot/4-shot >= 0pp')


## 7. Fixed Layer Configuration (Phase 1 V2)


In [ ]:
# ==============================================================================
# FIXED LAYER CONFIGURATION (Phase 1.5 — 10 Candidate Layers)
# ==============================================================================
# All 10 layers active simultaneously in Phase 1.5.
# Phase 2 (future): Router selects 0–4 best layers from these 10.
#
# Coverage for phi-2 (32 transformer layers):
#   Layer  8, 11  → early semantic + plan-parsing steering
#   Layer 13, 15  → proven effective layers (mid-range)
#   Layer 17, 19  → computation chain reinforcement
#   Layer 20, 21  → late-computation + answer assembly
#   Layer 22, 24  → output formatting + final anchor
#
# steer_amplitude_k=None  → amplitude = rho_max / sqrt(k=10) = 0.013 per layer
# steer_amplitude_k=3     → amplitude fixed at rho_max / sqrt(3) = 0.023 per layer
#   (Use 3 when you want per-layer amplitude identical to the Phase 1 3-layer case)
FIXED_LAYERS = [8, 11, 13, 15, 17, 19, 20, 21, 22, 24]
S_FIXED = ','.join(map(str, FIXED_LAYERS))
STEER_AMPLITUDE_K = None  # set to 3 for fixed per-layer amplitude

print('='*80)
print('✅ FIXED LAYER CONFIGURATION (Phase 1.5 — 10 Layers)')
print('='*80)
print(f'Active layers ({len(FIXED_LAYERS)}): {FIXED_LAYERS}')
print(f'S_FIXED argument: {S_FIXED}')
_amp = f'rho_max/sqrt({STEER_AMPLITUDE_K})={0.04/(STEER_AMPLITUDE_K**0.5):.4f}' \
       if STEER_AMPLITUDE_K else f'rho_max/sqrt(10)={0.04/10**0.5:.4f}'
print(f'Per-layer amplitude: {_amp}')
print(f'\n🔬 Cooperation: implicit via residual stream + typed L1 specialization')
print('='*80)


## 8. Stage S1: SteerNet Training (V2 Typed Loss)


In [ ]:
# Stage S1: SteerNet Training — 10 layers, typed L1, KL arithmetic, focal_gamma=2
import subprocess

# Build optional steer_amplitude_k arg
_amp_k_arg = f'--set steer_amplitude_k={STEER_AMPLITUDE_K}' if STEER_AMPLITUDE_K else ''

command = f'''python scripts/run_ars_pipeline.py --stage S1 {STEERNET_ARCH_ARGS} \
    --objective {S1_OBJECTIVE} \
    --s1_epochs {S1_EPOCHS} \
    --s1_lr {S1_LR} \
    --s1_train_n {S1_TRAIN_N} \
    --s1_val_n {S1_VAL_N} \
    --batch_size {BATCH_SIZE} \
    --steer_scale {STEER_SCALE} \
    --s_fixed {S_FIXED} \
    --set steer_kappa={STEER_KAPPA} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set grad_accum={GRAD_ACCUM} \
    --set max_tok_len={MAX_TOK_LEN} \
    --set loss_mode={LOSS_MODE} \
    {_amp_k_arg}'''

process = subprocess.Popen(
    command, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(
        f"S1 pipeline failed with exit code {process.returncode}; "
        "full runner output is shown above."
    )

print('\n' + '='*80)
print('✅ S1 Training Complete')
print('='*80)


## 9. Stage S5: Deployment Calibration (Contract v3)


In [ ]:
# Stage S5: Calibration — writes deployment_contract.json (version 3, no shot_scales)
!python scripts/run_ars_pipeline.py --stage S5 {STEERNET_ARCH_ARGS} \
    --calib_sel_n {CALIB_SEL_N} \
    --contract deployment_contract.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set k_scale_mode={K_SCALE_MODE}

import json as _json, pathlib as _pl
_c = _json.loads(_pl.Path("deployment_contract.json").read_text())
print(f"\nContract version: {_c.get('version')}  (expected 3)")
assert _c.get("version") == 3, "Expected contract v3 — shot_scales were not purged!"
assert "shot_scales" not in _c, "shot_scales field must not appear in v3 contract!"
print("\n" + "="*80)
print("✅ S5 Calibration Complete (Contract v3 verified)")
print("="*80)


## 10. Benchmark: Baseline vs Fixed-Control (Zero Heuristic Dials)


In [ ]:
# Benchmark: Baseline vs Fixed-control [13,20,22]
# Zero heuristic dials: model executes with identical weights and scale across ALL shots
!python scripts/run_ars_pipeline.py --stage bench {STEERNET_ARCH_ARGS} \
    --contract deployment_contract.json \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --record experiment_record.json \
    --report \
    --arms baseline,fixed \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE}

print('\n' + '='*80)
print('✅ Benchmark Complete')
print('='*80)


## 11. Inspect Benchmark Report


In [ ]:
import os, json
from IPython.display import display
from src.evaluation.diagnostics import plot_paired, plot_training, show_generations

# Report is now written to report/REPORT_GSM8K.md (suite-scoped) and report/REPORT.md (alias)
for report_path in ["report/REPORT_GSM8K.md", "report/REPORT.md"]:
    if os.path.exists(report_path):
        print(f"--- {report_path} ---")
        with open(report_path, "r", encoding="utf-8") as f:
            print(f.read())
        break
else:
    print("Report not found. Check benchmark logs above.")

if os.path.exists("report/training_log.json"):
    with open("report/training_log.json", encoding="utf-8") as f:
        hist = json.load(f)
    plot_training(hist, "report/training_curves.png")
    # Print V2 checkpoint metrics
    print(f"\nBest crit: {hist.get('best_crit', 'N/A')}")
    for ep_info in hist.get("epochs", []):
        if isinstance(ep_info, dict):
            print(f"  crit={ep_info.get('crit', '?'):.4f}  result_top1={ep_info.get('result_top1', '?'):.3f}  val_nll={ep_info.get('validation_nll', '?'):.4f}")

if os.path.exists("experiment_record.json"):
    with open("experiment_record.json", encoding="utf-8") as f:
        record = json.load(f)
    plot_paired(record, "report/paired_outcomes.png")
    # Print acceptance-criteria check
    print("\n\u2705 Acceptance Criteria Check:")
    shots_data = record.get("shots", {})
    for shot_str, shot_data in shots_data.items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        b_acc = b.get("lenient_accuracy", 0.0)
        s_acc = s.get("lenient_accuracy", 0.0)
        delta = s_acc - b_acc
        paired = shot_data.get("paired", {})
        print(f"  {shot_str}-shot | Baseline={b_acc:.1f}% | Fixed={s_acc:.1f}% | Delta={delta:+.1f} pp | Wins={paired.get('fixed_wins',0)} Losses={paired.get('fixed_losses',0)}")
        if paired.get("taxonomy"):
            print(f"    Taxonomy: {paired['taxonomy']}")
    print(json.dumps(record, indent=2))

# Per-arm generation files: gsm8k_{n}shot_{arm}.jsonl
for shot_str in shots_data.keys():
    gen_path = f"report/generations/gsm8k_{shot_str}shot_fixed.jsonl"
    if os.path.exists(gen_path):
        display(show_generations(gen_path, k=4, flips_only=False))


## 12. Save Artifacts


In [ ]:
import shutil, os, json

os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)
for fname in [
    "deployment_contract.json",
    "experiment_record.json",
    "norms_calibration.json",
    "checkpoints/steernet.pt",
]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
        print(f"Copied: {fname}")
if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
    print("Copied: report/")

# Verify contract version in artifact
_c_path = "/kaggle/working/ars_artifacts/deployment_contract.json"
if os.path.exists(_c_path):
    _c = json.loads(open(_c_path).read())
    print(f"\nArtifact contract version: {_c.get('version')}")
    assert _c.get("version") == 3, "Contract must be v3 in artifact!"
    assert "shot_scales" not in _c, "shot_scales must not be in artifact contract!"
    print("\u2705 Contract v3 artifact verified")

print("\nArtifacts saved to /kaggle/working/ars_artifacts:")
!ls -lh /kaggle/working/ars_artifacts
